# Notebook v6 — memisahkan detection headroom dari geometri kategori

Naskah mengakui satu keterbatasan yang menjadi sasaran utama reviewer: pada enam kategori,
F1 dasar dan laju intrusi sintetis mengurutkan kategori hampir identik (ρ = −0,94 di bawah
XGBoost), sehingga keduanya tidak bisa dibedakan. Notebook ini menyerang keterbatasan itu
dari dua arah.

**Bagian X — uji penentu.** Subsampel 29 sampel ditahan **persis sama**, lalu kekuatan
pengklasifikasi yang diubah: XGBoost dengan 10, 30, 100 dan 300 pohon, selebihnya identik
dengan naskah. Karena subsampelnya tidak berubah sedikit pun, laju intrusi ι, rasio
dispersi δ dan tipe Napierała **sama persis menurut konstruksi**, bukan sekadar mirip.
Kalau perolehan SMOTE tetap mengikuti headroom sementara ketiga ukuran geometri itu diam
di tempat, geometri tidak mungkin menjadi penjelasannya.

Kolom 300 pohon adalah konfigurasi naskah. Ia wajib mereproduksi hasil `secT_ablation.csv`
pada n = 29 baris per baris, dan sel analisis memeriksanya.

**Bagian Y — asal-usul kolinearitas.** ι dan δ diukur ulang pada subsampel berukuran 29,
50, 100 dan 500. Keduanya diperkirakan **turun** seiring n, dan bukan karena hubungan sebab:
ι membandingkan jarak titik sintetis ke minoritas terdekat dengan jarak ke kelas lain, dan
pembilang δ adalah jarak tetangga terdekat di dalam subsampel. Semakin banyak sampel,
semakin rapat awannya, semakin kecil kedua jarak itu. Mengukurnya mengubah kolinearitas
dari sesuatu yang diakui menjadi sesuatu yang dijelaskan.

Definisi `example_types`, `dispersion_ratio` dan `synthetic_intrusion` disalin apa adanya
dari notebook v3, sehingga angkanya sebanding langsung dengan Tabel 6 dan 9 naskah.

| Bagian | Isi | Perkiraan waktu |
|---|---|---|
| X | 240 model XGBoost pada empat tingkat kekuatan | ±2,5 jam |
| Y | geometri pada empat ukuran, tanpa melatih model | ±45 menit |
| Z | analisis gabungan | < 1 menit |

Notebook ini membaca `clean_data.npz` dan `secT_ablation.csv` yang sudah ada di
`MyDrive/juti_rerun`. Tidak ada yang diunduh ulang.


In [ ]:
# ====== PENGATURAN ======
N_FOLDS      = 5
TARGET       = 24000          # target penyeimbangan, identik dengan naskah
XGB_DEPTH    = 6
XGB_LR       = 0.1
RANDOM_STATE = 42
CTRL_N       = 29             # titik kontrol naskah; subsampel ditahan di sini
ABL_MIN_FRAC = 0.9            # Infiltration punya 28 sampel di satu lipatan

# --- Bagian X: ablasi kekuatan pengklasifikasi ---
CAP_TREES    = [10, 30, 100, 300]   # 300 = konfigurasi naskah, dipakai sebagai pemeriksa
CAP_SEEDS    = [42]
CAP_METHODS  = ["base", "smote"]

# --- Bagian Y: geometri pada beberapa ukuran subsampel ---
GEO_SIZES    = [29, 50, 100, 500]
N_SYNTH      = 2000           # identik dengan v3

TIME_BUDGET_MIN = 170         # berhenti rapi sebelum Colab memutus
RUN_X = True
RUN_Y = False                 # nyalakan setelah bagian X selesai

import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = "/content/drive/MyDrive/juti_rerun"
except Exception:
    OUTDIR = "./juti_rerun"
os.makedirs(OUTDIR, exist_ok=True)
print("Folder hasil:", OUTDIR)
print(f"Bagian X {'AKTIF' if RUN_X else 'mati'} | bagian Y {'AKTIF' if RUN_Y else 'mati'}")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn==1.6.1", "imbalanced-learn==0.14.2", "xgboost"], check=False)
import sklearn, imblearn, xgboost
print("scikit-learn", sklearn.__version__, "| imbalanced-learn", imblearn.__version__,
      "| xgboost", xgboost.__version__)

## 1. Baca data bersih dari cache

In [ ]:
CLEAN = os.path.join(OUTDIR, "clean_data.npz")
if not os.path.exists(CLEAN):
    raise SystemExit(
        "clean_data.npz tidak ditemukan di " + OUTDIR + "\n"
        "Jalankan notebook v3 bagian 0 sampai 3 lebih dulu.")
z = np.load(CLEAN, allow_pickle=True)
X_all = z["X_all"]
y_all = z["y_all"].astype(str)
CLASSES = sorted(np.unique(y_all))
TARGET_CATS = ["Botnet", "PortScan", "Web Attack", "Infiltration", "DoS", "DDoS"]
print(f"{X_all.shape[0]:,} record x {X_all.shape[1]} fitur | {len(CLASSES)} kelas")

CKT = os.path.join(OUTDIR, "secT_ablation.csv")
if os.path.exists(CKT):
    _t = pd.read_csv(CKT)
    print(f"secT_ablation.csv ditemukan: {len(_t)} baris, "
          f"ukuran {sorted(_t['size'].unique())}")
else:
    print("PERINGATAN: secT_ablation.csv tidak ada; bagian Z tidak bisa dijalankan.")

## X. Uji penentu: geometri ditahan, kekuatan pengklasifikasi diubah

Setiap lipatan, setiap kategori, subsampel 29 sampel yang **sama persis** dengan bagian T
notebook v5 — permutasi yang sama, benih yang sama. Yang berubah hanyalah jumlah pohon
XGBoost.

Karena subsampelnya tidak bergerak, ι, δ dan tipe Napierała kategori itu tetap pada nilai
yang sudah dilaporkan di Tabel 6 naskah. Maka setiap hubungan antara headroom dan perolehan
SMOTE yang muncul di dalam satu kategori **tidak mungkin** berasal dari geometri: geometri
tidak punya variasi untuk menjelaskan apa pun.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

CLS2ID = {c: i for i, c in enumerate(CLASSES)}
ID2CLS = np.array(CLASSES)

def fit_predict_xgb(trees, seed, Xs, ys, Xte):
    m = XGBClassifier(n_estimators=trees, max_depth=XGB_DEPTH, learning_rate=XGB_LR,
                      subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                      objective="multi:softmax", num_class=len(CLASSES),
                      n_jobs=-1, random_state=seed, verbosity=0)
    m.fit(Xs, np.array([CLS2ID[v] for v in ys], dtype=int))
    return ID2CLS[np.asarray(m.predict(Xte)).astype(int)]

def smote_strategy(y, target=TARGET):
    vc = pd.Series(y).value_counts()
    return {c: target for c, n in vc.items() if n < target}

def counts_for(y_true, y_pred, c):
    return (int(((y_pred == c) & (y_true == c)).sum()),
            int(((y_pred == c) & (y_true != c)).sum()),
            int(((y_pred != c) & (y_true == c)).sum()))

CKX = os.path.join(OUTDIR, "secX_capacity.csv")
COLS_X = ["seed", "fold", "category", "trees", "method", "n_train",
          "tp", "fp", "fn", "fit_seconds"]
doneX = pd.read_csv(CKX) if os.path.exists(CKX) else pd.DataFrame(columns=COLS_X)
keysX = set(zip(doneX.seed, doneX.fold, doneX.category, doneX.trees,
                doneX.method)) if len(doneX) else set()
TOTAL_X = len(CAP_SEEDS) * N_FOLDS * len(TARGET_CATS) * len(CAP_TREES) * len(CAP_METHODS)
print(f"Total {TOTAL_X} model | sudah ada {len(keysX)}")

if not RUN_X:
    print("\n>>> RUN_X = False, jadi tidak ada model yang dilatih di sel ini.")
    print(">>> Ubah RUN_X = True di sel Pengaturan, jalankan ulang sel itu, lalu sel ini.")

t_start, n_run, stopped = time.time(), 0, False
for seed in (CAP_SEEDS if RUN_X else []):
    if stopped: break
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(skf.split(X_all, y_all)):
        if stopped: break
        if not any((seed, fold, c, tn, me) not in keysX
                   for c in TARGET_CATS for tn in CAP_TREES for me in CAP_METHODS):
            continue
        sc = StandardScaler().fit(X_all[tr])
        Xtr_f, Xte_f = sc.transform(X_all[tr]), sc.transform(X_all[te])
        ytr_f, yte_f = y_all[tr], y_all[te]
        for c in TARGET_CATS:
            if stopped: break
            idx_all = np.where(ytr_f == c)[0]
            n_eff = min(CTRL_N, len(idx_all))
            if n_eff < ABL_MIN_FRAC * CTRL_N:
                continue
            # permutasi identik dengan bagian T notebook v5
            perm = np.random.default_rng(seed * 100 + fold).permutation(idx_all)
            idx_c = perm[:n_eff]
            keep_idx = np.r_[np.where(ytr_f != c)[0], idx_c]
            Xk, yk = Xtr_f[keep_idx], ytr_f[keep_idx]
            for me in CAP_METHODS:
                if stopped: break
                todo = [tn for tn in CAP_TREES if (seed, fold, c, tn, me) not in keysX]
                if not todo:
                    continue
                Xs, ys = ((SMOTE(sampling_strategy=smote_strategy(yk), k_neighbors=5,
                                 random_state=seed).fit_resample(Xk, yk))
                          if me == "smote" else (Xk, yk))
                for tn in todo:
                    if TIME_BUDGET_MIN and (time.time()-t_start)/60 >= TIME_BUDGET_MIN:
                        stopped = True; break
                    t0 = time.time()
                    pred = fit_predict_xgb(tn, seed, Xs, ys, Xte_f)
                    dt = time.time() - t0
                    tp, fp, fn = counts_for(yte_f, pred, c)
                    doneX = pd.concat([doneX, pd.DataFrame([dict(
                        seed=seed, fold=fold, category=c, trees=tn, method=me,
                        n_train=len(idx_c), tp=tp, fp=fp, fn=fn,
                        fit_seconds=round(dt, 1))])], ignore_index=True)
                    doneX.to_csv(CKX, index=False)
                    keysX.add((seed, fold, c, tn, me)); n_run += 1
                    left = TOTAL_X - len(keysX)
                    avg = (time.time()-t_start)/max(n_run, 1)
                    print(f"seed {seed} fold {fold} {c:12s} {tn:>3} pohon {me:5s} "
                          f"{dt:5.0f} s | sisa {left} ≈ {left*avg/60:.0f} menit")

if stopped:
    print("\nBerhenti karena batas waktu. Jalankan ulang sel ini untuk melanjutkan.")
elif RUN_X and len(keysX) >= TOTAL_X:
    print("\nBagian X selesai. Jalankan sel analisis di bawah, "
          "lalu nyalakan RUN_Y untuk bagian Y.")

## X2. Analisis: apakah headroom memprediksi perolehan ketika geometri diam?

Tiga hal yang diperiksa sel ini.

Pertama, **pemeriksaan reproduksi**: kolom 300 pohon harus sama persis dengan hasil XGBoost
pada n = 29 di `secT_ablation.csv`. Kalau tidak, ada yang berbeda di antara kedua notebook
dan sisa analisisnya tidak bisa dipercaya.

Kedua, **hubungan di dalam kategori**: untuk setiap kategori, korelasi antara headroom dan
perolehan di keempat tingkat kekuatan. Di sini ι dan δ konstan menurut konstruksi.

Ketiga, **hubungan lintas kisi**: 24 sel (enam kategori × empat tingkat kekuatan), dengan
nilai p dari uji permutasi, bukan pendekatan asimtotik.


In [ ]:
from scipy.stats import spearmanr, wilcoxon
f1 = lambda tp, fp, fn: 200.0*tp/max(2*tp+fp+fn, 1)
X = pd.read_csv(CKX)
if not len(X):
    raise SystemExit("Bagian X belum menghasilkan apa pun. Nyalakan RUN_X dan jalankan sel X.")

rows = []
for (c, tn), sub in X.groupby(["category", "trees"]):
    b = sub[sub.method == "base"].set_index(["seed", "fold"]).sort_index()
    s = sub[sub.method == "smote"].set_index(["seed", "fold"]).sort_index()
    k = b.index.intersection(s.index)
    if not len(k):
        continue
    b, s = b.loc[k], s.loc[k]
    fb = np.array([f1(*r) for r in b[["tp", "fp", "fn"]].values])
    fs = np.array([f1(*r) for r in s[["tp", "fp", "fn"]].values])
    rows.append(dict(category=c, trees=int(tn), n_pairs=len(k),
                     base_F1=fb.mean(), smote_F1=fs.mean(), delta=(fs-fb).mean(),
                     headroom=100-fb.mean(),
                     R_c=100.0*(b.fn.sum()-s.fn.sum())/max(b.fn.sum(), 1)))
A = pd.DataFrame(rows)
ORDER = [c for c in TARGET_CATS if c in set(A.category)]

REF_TREES = max(CAP_TREES)
print(f"=== 1. pemeriksaan reproduksi: {REF_TREES} pohon vs secT_ablation.csv pada n = {CTRL_N} ===")
if os.path.exists(CKT):
    t = pd.read_csv(CKT)
    t = t[(t.skipped == 0) & (t["size"] == CTRL_N) & (t.model == "xgb")]
    ok = True
    for c in ORDER:
        sub = t[t.category == c]
        if not len(sub):
            continue
        bb = sub[sub.method == "base"].sort_values("fold")
        ss = sub[sub.method == "smote"].sort_values("fold")
        fb = np.mean([f1(*r) for r in bb[["tp", "fp", "fn"]].values])
        fs = np.mean([f1(*r) for r in ss[["tp", "fp", "fn"]].values])
        mine = A[(A.category == c) & (A.trees == REF_TREES)]
        if not len(mine):
            continue
        db, dd = abs(mine.base_F1.iloc[0]-fb), abs(mine.delta.iloc[0]-(fs-fb))
        flag = "" if max(db, dd) < 1e-9 else "   <-- TIDAK COCOK"
        ok &= max(db, dd) < 1e-9
        print(f"  {c:13s} selisih base {db:.2e}, selisih delta {dd:.2e}{flag}")
    print("  -> " + ("identik, kedua notebook sepakat" if ok else
                     "ADA SELISIH: periksa sebelum melanjutkan"))
else:
    print("  (secT_ablation.csv tidak ada)")

print("\n=== 2. F1 dasar dan perolehan menurut kekuatan pengklasifikasi ===")
for val, lab in [("base_F1", "F1 tanpa resampling"), ("delta", "delta F1 dari SMOTE"),
                 ("R_c", "pangsa serangan terlewat yang dipulihkan (%)")]:
    p = A.pivot_table(index="category", columns="trees", values=val).reindex(ORDER)
    print(f"\n{lab}:")
    print(p.round(2).to_string())

print("\n\n=== 3. di dalam kategori: headroom vs perolehan, geometri konstan ===")
print("    (iota, delta dan tipe Napierala identik di keempat kolom menurut konstruksi)")
rs = []
for c in ORDER:
    s = A[A.category == c].sort_values("trees")
    if len(s) < 3 or s.base_F1.nunique() < 2:
        print(f"  {c:13s} dilewati: butuh minimal tiga tingkat kekuatan "
              f"dengan F1 dasar yang berbeda (ada {len(s)})")
        continue
    r = spearmanr(s.headroom, s.delta).statistic
    rs.append(r)
    print(f"  {c:13s} rho = {r:+.2f}   headroom: " +
          " -> ".join(f"{v:5.1f}" for v in s.headroom) +
          "   delta: " + " -> ".join(f"{v:6.2f}" for v in s.delta))
if rs:
    print(f"\n  median rho di dalam kategori = {np.median(rs):+.2f} "
          f"({sum(r > 0 for r in rs)} dari {len(rs)} kategori positif)")

print("\n=== 4. lintas seluruh kisi (uji permutasi) ===")
rng = np.random.default_rng(0)
r = spearmanr(A.headroom, A.delta)
null = np.array([spearmanr(A.headroom, rng.permutation(A.delta.values)).statistic
                 for _ in range(100000)])
p = (np.sum(np.abs(null) >= abs(r.statistic)) + 1) / (len(null) + 1)
print(f"  n = {len(A)} sel: rho = {r.statistic:+.2f} (p permutasi = {p:.4f})")

A.to_csv(os.path.join(OUTDIR, "v6_capacity_summary.csv"), index=False)
print("\nDisimpan: v6_capacity_summary.csv")

## Y. Dari mana kolinearitas itu berasal

ι dan δ diukur pada subsampel bersarang berukuran 29, 50, 100 dan 500 — subsampel yang sama
persis dengan bagian T notebook v5. Tidak ada model yang dilatih di sini.

Dugaannya: keduanya turun seiring n, karena definisinya memuat jarak di dalam subsampel, dan
awan yang lebih padat memberi jarak yang lebih kecil. Kalau dugaan itu benar, kolinearitas
antara headroom dan geometri di naskah punya penjelasan mekanis, bukan sekadar kebetulan
enam kategori.


In [ ]:
from sklearn.neighbors import NearestNeighbors

K_NN = 5

def example_types(X_min, X_other, k=K_NN):
    X_both = np.vstack([X_min, X_other])
    is_min = np.r_[np.ones(len(X_min), bool), np.zeros(len(X_other), bool)]
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(X_both).kneighbors(X_min)
    same = is_min[ind[:, 1:]].sum(axis=1)
    return {"safe (%)": np.mean(same >= 4) * 100,
            "borderline (%)": np.mean((same == 2) | (same == 3)) * 100,
            "rare (%)": np.mean(same == 1) * 100,
            "outlier (%)": np.mean(same == 0) * 100}

def dispersion_ratio(X_min, nn_other):
    d_in, _ = NearestNeighbors(n_neighbors=2).fit(X_min).kneighbors(X_min)
    d_out, _ = nn_other.kneighbors(X_min, n_neighbors=1)
    return d_in[:, 1].mean() / max(d_out[:, 0].mean(), 1e-12)

def synthetic_intrusion(X_min, nn_other, rng, k=K_NN, n_new=N_SYNTH):
    k = min(k, len(X_min) - 1)
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(X_min)
    i = rng.integers(0, len(X_min), n_new)
    j = ind[i, rng.integers(1, k + 1, n_new)]
    lam = rng.random((n_new, 1))
    X_new = X_min[i] + lam * (X_min[j] - X_min[i])
    d_min, _ = NearestNeighbors(n_neighbors=1).fit(X_min).kneighbors(X_new)
    d_oth, _ = nn_other.kneighbors(X_new, n_neighbors=1)
    return np.mean(d_oth[:, 0] < d_min[:, 0]) * 100

CKY = os.path.join(OUTDIR, "secY_geometry_by_size.csv")
COLS_Y = ["seed", "fold", "category", "size", "n_used", "safe (%)", "borderline (%)",
          "rare (%)", "outlier (%)", "dispersion ratio", "synthetic intrusion (%)", "seconds"]
doneY = pd.read_csv(CKY) if os.path.exists(CKY) else pd.DataFrame(columns=COLS_Y)
keysY = set(zip(doneY.seed, doneY.fold, doneY.category, doneY["size"])) if len(doneY) else set()

if not RUN_Y:
    print("\n>>> RUN_Y = False, jadi tidak ada yang dihitung di sel ini.")
    print(">>> Ubah RUN_Y = True di sel Pengaturan, jalankan ulang sel itu, lalu sel ini.")

t_start, stopped = time.time(), False
for seed in (CAP_SEEDS if RUN_Y else []):
    if stopped: break
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, _) in enumerate(skf.split(X_all, y_all)):
        if stopped: break
        if not any((seed, fold, c, n) not in keysY
                   for c in TARGET_CATS for n in GEO_SIZES):
            continue
        sc = StandardScaler().fit(X_all[tr])
        Xtr_f, ytr_f = sc.transform(X_all[tr]), y_all[tr]
        for c in TARGET_CATS:
            if stopped: break
            if not any((seed, fold, c, n) not in keysY for n in GEO_SIZES):
                continue
            idx_all = np.where(ytr_f == c)[0]
            perm = np.random.default_rng(seed * 100 + fold).permutation(idx_all)
            X_oth = Xtr_f[ytr_f != c]
            nn_o = NearestNeighbors(n_neighbors=1).fit(X_oth)
            for n in GEO_SIZES:
                if (seed, fold, c, n) in keysY:
                    continue
                if TIME_BUDGET_MIN and (time.time()-t_start)/60 >= TIME_BUDGET_MIN:
                    stopped = True; break
                n_eff = min(n, len(idx_all))
                if n_eff < ABL_MIN_FRAC * n:
                    continue                      # ukuran ini mustahil bagi kategori ini
                t0 = time.time()
                X_min = Xtr_f[perm[:n_eff]]
                row = dict(seed=seed, fold=fold, category=c, size=n, n_used=n_eff,
                           **example_types(X_min, X_oth))
                row["dispersion ratio"] = dispersion_ratio(X_min, nn_o)
                row["synthetic intrusion (%)"] = synthetic_intrusion(
                    X_min, nn_o, np.random.default_rng(seed + fold))
                row["seconds"] = round(time.time() - t0, 1)
                doneY = pd.concat([doneY, pd.DataFrame([row])], ignore_index=True)
                doneY.to_csv(CKY, index=False)
                keysY.add((seed, fold, c, n))
                print(f"seed {seed} fold {fold} {c:12s} n={n:<4} "
                      f"iota={row['synthetic intrusion (%)']:5.1f}%  "
                      f"delta={row['dispersion ratio']:.2f}  {row['seconds']:4.0f} s")

if stopped:
    print("\nBerhenti karena batas waktu. Jalankan ulang sel ini untuk melanjutkan.")
elif RUN_Y:
    print("\nBagian Y selesai.")

## Z. Analisis gabungan

Menyatukan perolehan SMOTE dari bagian T notebook v5 dengan geometri bagian Y pada
subsampel yang sama, lalu menanyakan: ketika headroom dan geometri sama-sama tersedia di
atas kisi 5 kategori × 4 ukuran, mana yang masih memprediksi perolehan setelah yang lain
diperhitungkan?

Korelasi parsial dihitung atas peringkat, jadi ia adalah versi parsial dari Spearman.


In [ ]:
from scipy.stats import spearmanr, rankdata
f1 = lambda tp, fp, fn: 200.0*tp/max(2*tp+fp+fn, 1)

CKY = os.path.join(OUTDIR, "secY_geometry_by_size.csv")
if not os.path.exists(CKY) or not os.path.exists(CKT):
    raise SystemExit("Bagian Z butuh secY_geometry_by_size.csv dan secT_ablation.csv.")
Y = pd.read_csv(CKY)
T = pd.read_csv(CKT)
T = T[T.skipped == 0].copy()
T["f1"] = [f1(*r) for r in T[["tp", "fp", "fn"]].values]

geo = Y.groupby(["category", "size"])[["dispersion ratio", "synthetic intrusion (%)"]].mean()
print("=== 1. apakah geometri bergeser seiring ukuran subsampel? ===")
for col, lab in [("synthetic intrusion (%)", "laju intrusi sintetis iota (%)"),
                 ("dispersion ratio", "rasio dispersi delta")]:
    p = geo[col].unstack()
    p = p.reindex([c for c in TARGET_CATS if c in p.index])
    print(f"\n{lab}:")
    print(p.round(3).to_string())
    sub = p.dropna()
    if len(sub.columns) >= 3:
        rs, flat = [], []
        for c in sub.index:
            if sub.loc[c].nunique() < 2:
                flat.append(c); continue
            rs.append(spearmanr(list(sub.columns), sub.loc[c]).statistic)
        if rs:
            print(f"  rho(ukuran, nilai) per kategori: " + ", ".join(f"{v:+.2f}" for v in rs)
                  + f"   median {np.median(rs):+.2f}  ({sum(r < 0 for r in rs)}/{len(rs)} menurun)")
        if flat:
            print(f"  datar sepenuhnya, tidak bisa dikorelasikan: {', '.join(flat)}")

rows = []
for (c, n, mo), sub in T.groupby(["category", "size", "model"]):
    b = sub[sub.method == "base"].set_index(["seed", "fold"]).sort_index()
    s = sub[sub.method == "smote"].set_index(["seed", "fold"]).sort_index()
    k = b.index.intersection(s.index)
    if not len(k) or (c, n) not in geo.index:
        continue
    b, s = b.loc[k], s.loc[k]
    rows.append(dict(category=c, size=int(n), model=mo,
                     headroom=100-b.f1.mean(), delta=(s.f1.values-b.f1.values).mean(),
                     iota=geo.loc[(c, n), "synthetic intrusion (%)"],
                     disp=geo.loc[(c, n), "dispersion ratio"]))
G = pd.DataFrame(rows)

def partial(x, y, z):
    """Korelasi Spearman parsial antara x dan y setelah z dikendalikan."""
    rx, ry, rz = (rankdata(v) for v in (x, y, z))
    rz = np.c_[np.ones(len(rz)), rz]
    ex = rx - rz @ np.linalg.lstsq(rz, rx, rcond=None)[0]
    ey = ry - rz @ np.linalg.lstsq(rz, ry, rcond=None)[0]
    return float(np.corrcoef(ex, ey)[0, 1])

print("\n\n=== 2. headroom dan iota pada kisi kategori x ukuran ===")
for mo in sorted(G.model.unique()):
    s = G[G.model == mo]
    print(f"\n{mo.upper()} ({len(s)} sel):")
    print(f"  headroom vs delta       rho = {spearmanr(s.headroom, s.delta).statistic:+.2f}")
    print(f"  iota     vs delta       rho = {spearmanr(s.iota, s.delta).statistic:+.2f}")
    print(f"  headroom vs iota        rho = {spearmanr(s.headroom, s.iota).statistic:+.2f}"
          "   <-- kolinearitas")
    print(f"  headroom vs delta | iota  = {partial(s.headroom, s.delta, s.iota):+.2f}")
    print(f"  iota     vs delta | headroom = {partial(s.iota, s.delta, s.headroom):+.2f}")

G.to_csv(os.path.join(OUTDIR, "v6_grid_summary.csv"), index=False)
print("\nDisimpan: v6_grid_summary.csv")